In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

def fit_model():
    df = pd.read_csv('data/00000.csv')
    
    # Process data as in tinyphysics.py
    roll_lataccel = np.sin(df['roll'].values) * 9.81
    v_ego = df['vEgo'].values
    a_ego = df['aEgo'].values
    target_lataccel = df['targetLateralAcceleration'].values
    steer_command = -df['steerCommand'].values # Negate as per tinyphysics.py
    
    # We want to predict next lat accel based on current state and action.
    # In the dataset, we don't have "current_lataccel" separate from "target".
    # However, the simulator treats the dataset's target as the ground truth for training?
    # The README says "This is a 'simulated car' that has been trained to mimic... given realistic driving noise."
    # The dataset is "actual car and road states".
    # So 'targetLateralAcceleration' in the CSV is likely the MEASURED lateral acceleration of the car.
    # Let's assume target_lataccel is the actual lat accel for fitting.
    
    # y[t+1]
    y_next = target_lataccel[1:]
    
    # Features at t
    y_curr = target_lataccel[:-1]
    u_curr = steer_command[:-1]
    v_curr = v_ego[:-1]
    roll_curr = roll_lataccel[:-1]
    
    # Model: y_next = c1 * y_curr + c2 * u_curr * v_curr^2 + c3 * roll_curr + c4
    X = np.column_stack([
        y_curr,
        u_curr * (v_curr**2),
        roll_curr
    ])
    
    model = LinearRegression()
    model.fit(X, y_next)
    
    print("Coefficients:", model.coef_)
    print("Intercept:", model.intercept_)
    print("Score:", model.score(X, y_next))
    
    return model

fit_model()

In [ ]:
from tinyphysics import TinyPhysicsModel, TinyPhysicsSimulator, CONTROL_START_IDX
from controllers import pid
from matplotlib import pyplot as plt
import seaborn as sns

sns.set_theme()


In [ ]:
def plot_rollout(sim):
  fig, ax = plt.subplots(figsize=(10, 5))
  ax.plot(sim.target_lataccel_history, label="Target Lateral Acceleration", alpha=0.5)
  ax.plot(sim.current_lataccel_history, label="Actual Lateral Acceleration", alpha=0.5)
  ax.legend()
  ax.set_xlabel("Step")
  ax.set_ylabel("Lateral Acceleration")
  ax.set_title("Rollout")
  plt.show()

In [ ]:
model = TinyPhysicsModel("./models/tinyphysics.onnx", debug=True)
controller = pid.Controller()

In [ ]:
sim = TinyPhysicsSimulator(model, "./data/00000.csv", controller=controller, debug=False)
sim.rollout()

In [ ]:
plot_rollout(sim)